# NSE Build Diagnostics & Auto-Fix

> Works on iPhone/iPad via Colab. Go to colab.research.google.com -> File -> Upload notebook -> pick this file.

| Cell | Module | What it does |
|------|--------|--------------|
| 1 | Install deps | PyNaCl, GitPython, python-frontmatter, tabulate |
| 2 | Config | Paste PAT here - only cell requiring input |
| 3 | CI status | Latest Actions run per workflow across all repos |
| 4 | Log tailing | Download failed job logs, extract error lines |
| 5 | Dep audit | Scan package.json / requirements.txt for issues |
| 6 | Workflow lint | Schema-validate all .github/workflows yml files |
| 7 | Cross-repo scan | Build health dashboard across all repos |
| 8 | Auto-fix | Pattern-match findings, emit exact fix commands |
| 9 | Health report | Write and push markdown report to nse-kb |
| 10 | Rerun failed | Trigger rerun of failed CI jobs via API |

**Run Cell 1, then Cell 2 (fill in PAT), then any cell in any order.**

In [ ]:
# Cell 1 - Install all dependencies
import subprocess, sys
pkgs = ["PyNaCl", "python-frontmatter", "GitPython",
        "requests", "PyYAML", "packaging", "tabulate"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

import os, re, json, base64, time, shutil
from pathlib import Path
from datetime import datetime, timezone
import requests, yaml, frontmatter
import git as gitmod
from tabulate import tabulate

print("All dependencies installed.")

In [ ]:
# Cell 2 - FILL IN GITHUB_PAT THEN RUN THIS CELL
GITHUB_PAT   = ""          # paste your fine-grained PAT here
PRIMARY_REPO = "qt314wink/nse-kb"
BRANCH       = "main"
LOCAL_DIR    = "/content/nse-kb"

EXTRA_REPOS = [
    "qt314wink/omni-loom-case-study",
    "qt314wink/cicd-mastery-platform",
    "qt314wink/philly-civic-ai",
    "qt314wink/skills-taxonomy-pipeline",
    "qt314wink/nova-digital",
    "qt314wink/mochi-ui",
]
ALL_REPOS = [PRIMARY_REPO] + EXTRA_REPOS

API = "https://api.github.com"
H = {
    "Authorization": f"Bearer {GITHUB_PAT}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}
NOW       = datetime.now(timezone.utc)
TIMESTAMP = NOW.strftime("%Y-%m-%dT%H:%M:%SZ")
DATE_SLUG = NOW.strftime("%Y-%m-%d")

if not GITHUB_PAT:
    raise ValueError("GITHUB_PAT is empty - paste your token above.")

me = requests.get(f"{API}/user", headers=H)
assert me.status_code == 200, f"Auth failed: HTTP {me.status_code}"
GH_USER = me.json()["login"]

repo_meta = requests.get(f"{API}/repos/{PRIMARY_REPO}", headers=H).json()
assert repo_meta.get("permissions", {}).get("push"), "PAT has no push access"

print(f"Authenticated as: {GH_USER}")
print(f"Push access confirmed: {PRIMARY_REPO}")

STATUS_ICON = {
    "success": "OK", "failure": "FAIL", "cancelled": "CANCELLED",
    "skipped": "SKIP", "in_progress": "RUNNING", "queued": "QUEUED",
    "neutral": "NEUTRAL", None: "?"
}

def get_file(repo, path):
    r = requests.get(f"{API}/repos/{repo}/contents/{path}", headers=H)
    if r.status_code == 200:
        return base64.b64decode(r.json()["content"]).decode("utf-8", errors="replace")
    return None

def get_dir_files(repo, path):
    r = requests.get(f"{API}/repos/{repo}/contents/{path}", headers=H)
    if r.status_code == 200 and isinstance(r.json(), list):
        return r.json()
    return []

def get_workflow_runs(repo, per_page=5):
    r = requests.get(f"{API}/repos/{repo}/actions/runs",
        headers=H, params={"per_page": per_page, "branch": BRANCH})
    if r.status_code != 200:
        return None, r.status_code
    return r.json().get("workflow_runs", []), 200

FAILED_RUNS  = []
AUDIT_TABLE  = []
LINT_RESULTS = []
BUILD_HEALTH = []
ALL_RUNS     = []
SUGGESTIONS  = []

In [ ]:
# Cell 3 - CI Run Status across all repos
FAILED_RUNS.clear()
ALL_RUNS.clear()

for repo in ALL_REPOS:
    runs, status = get_workflow_runs(repo)
    if runs is None:
        print(f"  {repo}: HTTP {status} (no access or no Actions)")
        continue
    if not runs:
        print(f"  {repo}: no runs on {BRANCH}")
        continue
    seen = {}
    for run in runs:
        wf = run["name"]
        if wf not in seen:
            seen[wf] = run
    for wf, run in seen.items():
        conclusion = run.get("conclusion")
        icon = STATUS_ICON.get(conclusion, "?")
        row = [
            repo.split("/")[1], wf, icon,
            run["head_sha"][:7],
            run["created_at"][:16].replace("T", " "),
            run["html_url"]
        ]
        ALL_RUNS.append(row)
        if conclusion == "failure":
            FAILED_RUNS.append({"repo": repo, "run": run})

print(tabulate(ALL_RUNS,
    headers=["Repo", "Workflow", "Result", "SHA", "Started", "URL"],
    tablefmt="rounded_outline", maxcolwidths=[20, 30, 12, 8, 17, 60]))

if FAILED_RUNS:
    print(f"\n{len(FAILED_RUNS)} failed run(s). Run Cell 4 to inspect logs.")
else:
    print("\nAll clear.")

In [ ]:
# Cell 4 - CI Log Tailing (extract errors from failed jobs)
ERR_RE = re.compile(
    r"error[: ]|FAIL |failed|Cannot find module|ModuleNotFoundError|"
    r"ImportError|SyntaxError|TypeError|Permission denied|"
    r"refusing to allow|non-zero exit|exit code [1-9]|"
    r"npm ERR!|pip.*ERROR|missing.*field|invalid.*value",
    re.I
)

def fetch_job_logs(repo, run_id):
    r = requests.get(f"{API}/repos/{repo}/actions/runs/{run_id}/jobs", headers=H)
    if r.status_code != 200:
        return []
    jobs = [j for j in r.json().get("jobs", []) if j.get("conclusion") == "failure"]
    results = []
    for job in jobs:
        log_r = requests.get(
            f"{API}/repos/{repo}/actions/jobs/{job['id']}/logs",
            headers=H, allow_redirects=True)
        if log_r.status_code == 200:
            error_lines = [
                l.strip() for l in log_r.text.splitlines()
                if ERR_RE.search(l) and len(l.strip()) > 5
            ][:30]
            results.append({
                "job": job["name"],
                "errors": error_lines,
                "log_url": job["html_url"]
            })
    return results

if not FAILED_RUNS:
    print("No failed runs to inspect.")
else:
    for item in FAILED_RUNS:
        repo, run = item["repo"], item["run"]
        print(f"\n{'='*60}")
        print(f"FAIL: {repo} -> {run['name']}")
        print(f"  {run['html_url']}")
        for jr in fetch_job_logs(repo, run["id"]):
            print(f"\n  Job: {jr['job']}")
            print(f"  Logs: {jr['log_url']}")
            for line in (jr["errors"] or ["(no matching error lines)"]):
                print(f"    > {line[:120]}")

In [ ]:
# Cell 5 - Dependency Audit (package.json + requirements.txt)
AUDIT_TABLE.clear()

for repo in ALL_REPOS:
    short = repo.split("/")[1]
    pkg = get_file(repo, "package.json")
    if pkg:
        try:
            pkgjson = json.loads(pkg)
            for section in ["dependencies", "devDependencies"]:
                for dep, ver in (pkgjson.get(section) or {}).items():
                    issues = []
                    if ver == "*":             issues.append("wildcard (*)")
                    if ver == "latest":        issues.append("unpinned (latest)")
                    if ver.startswith("^"):    issues.append("caret range (^)")
                    if ver.startswith("~"):    issues.append("tilde range (~)")
                    if issues:
                        AUDIT_TABLE.append([short, "package.json", dep, ver, ", ".join(issues)])
        except json.JSONDecodeError:
            AUDIT_TABLE.append([short, "package.json", "PARSE ERROR", "", "Invalid JSON"])

    req = get_file(repo, "requirements.txt")
    if req:
        for line in req.splitlines():
            line = line.strip()
            if not line or line.startswith("#"): continue
            if not any(op in line for op in ["==", ">=", "<="]):
                AUDIT_TABLE.append([short, "requirements.txt", line, "", "unpinned"])

    has_pkg  = pkg is not None
    has_lock = any(
        requests.get(f"{API}/repos/{repo}/contents/{lf}", headers=H).status_code == 200
        for lf in ["package-lock.json", "yarn.lock", "pnpm-lock.yaml"]
    )
    if has_pkg and not has_lock:
        AUDIT_TABLE.append([short, "lock file", "--", "", "no lock file"])

if AUDIT_TABLE:
    print(tabulate(AUDIT_TABLE,
        headers=["Repo", "File", "Package", "Version", "Issue"],
        tablefmt="rounded_outline"))
    print(f"\n{len(AUDIT_TABLE)} issue(s) found.")
else:
    print("No dependency issues found.")

In [ ]:
# Cell 6 - Workflow YAML Lint
LINT_RESULTS.clear()

DEPRECATED = {
    "actions/checkout@v2": "use @v4",
    "actions/checkout@v3": "use @v4",
    "actions/setup-python@v3": "use @v5",
    "actions/setup-python@v4": "use @v5",
    "actions/setup-node@v2": "use @v4",
    "actions/setup-node@v3": "use @v4",
    "actions/cache@v2": "use @v4",
    "actions/cache@v3": "use @v4",
}
HARDCODED_RE = re.compile(
    r"(password|api_key|secret|token)\s*[:=]\s*[\"'][^$]", re.I)

for repo in ALL_REPOS:
    short = repo.split("/")[1]
    for wf_meta in get_dir_files(repo, ".github/workflows"):
        if not wf_meta["name"].endswith(".yml"):
            continue
        content = get_file(repo, f".github/workflows/{wf_meta['name']}")
        if not content:
            continue
        wf_name = wf_meta["name"]
        link = f"https://github.com/{repo}/blob/{BRANCH}/.github/workflows/{wf_name}"
        try:
            data = yaml.safe_load(content) or {}
        except yaml.YAMLError as e:
            LINT_RESULTS.append([short, wf_name, f"YAML parse error: {e}", link])
            continue

        issues = []
        if "name" not in data:           issues.append("missing name:")
        if not data.get("jobs"):         issues.append("no jobs defined")
        if "permissions" not in data:    issues.append("no permissions block")
        for old, fix in DEPRECATED.items():
            if old in content:           issues.append(f"deprecated {old} -> {fix}")
        if HARDCODED_RE.search(content): issues.append("HARDCODED CREDENTIAL DETECTED")

        if issues:
            for issue in issues:
                LINT_RESULTS.append([short, wf_name, issue, link])
        else:
            LINT_RESULTS.append([short, wf_name, "clean", ""])

print(tabulate(LINT_RESULTS,
    headers=["Repo", "Workflow", "Finding", "Link"],
    tablefmt="rounded_outline", maxcolwidths=[18, 30, 55, 60]))

In [ ]:
# Cell 7 - Cross-Repo Build Health Dashboard
BUILD_HEALTH.clear()

for repo in ALL_REPOS:
    short = repo.split("/")[1]
    r = requests.get(f"{API}/repos/{repo}", headers=H)
    if r.status_code != 200:
        BUILD_HEALTH.append([short, "--", "no access", "--", "--", "--"])
        continue
    meta = r.json()
    pushed = meta.get("pushed_at", "")[:10]
    issues = meta.get("open_issues_count", 0)
    branch = meta.get("default_branch", "main")

    runs, _ = get_workflow_runs(repo, per_page=3)
    if runs:
        ci = STATUS_ICON.get(runs[0].get("conclusion"), "?") + " " + (runs[0].get("conclusion") or runs[0].get("status","?"))
    else:
        ci = "no runs"

    try:
        pushed_dt = datetime.strptime(pushed, "%Y-%m-%d").replace(tzinfo=timezone.utc)
        days = (NOW - pushed_dt).days
        stale = f"STALE {days}d" if days > 30 else f"{days}d ago"
    except:
        stale = pushed

    missing = [f for f in ["README.md", ".gitignore"]
               if requests.get(f"{API}/repos/{repo}/contents/{f}", headers=H).status_code != 200]

    BUILD_HEALTH.append([short, branch, ci, stale, issues, ", ".join(missing) or "ok"])

print(tabulate(BUILD_HEALTH,
    headers=["Repo", "Branch", "CI", "Last Push", "Open Issues", "Missing Files"],
    tablefmt="rounded_outline"))

In [ ]:
# Cell 8 - Auto-Fix Suggestions
SUGGESTIONS.clear()

FIX_PATTERNS = [
    (r"wildcard",
     "Pin wildcard dependency",
     "npm install <pkg>@latest --save-exact"),
    (r"unpinned.*latest",
     "Replace latest with pinned semver",
     "npm install <pkg>@<version> --save-exact"),
    (r"caret range",
     "Caret allows minor bumps - pin if stability is critical",
     "Change \"^1.2.3\" to \"1.2.3\" in package.json"),
    (r"no lock file",
     "Generate and commit a lock file",
     "npm install\ngit add package-lock.json\ngit commit -m 'chore: add lock file'"),
    (r"checkout@v[23]",
     "Upgrade actions/checkout to v4",
     "sed -i 's|actions/checkout@v[23]|actions/checkout@v4|g' .github/workflows/*.yml"),
    (r"setup-python@v[34]",
     "Upgrade actions/setup-python to v5",
     "sed -i 's|setup-python@v[34]|setup-python@v5|g' .github/workflows/*.yml"),
    (r"setup-node@v[23]",
     "Upgrade actions/setup-node to v4",
     "sed -i 's|setup-node@v[23]|setup-node@v4|g' .github/workflows/*.yml"),
    (r"cache@v[23]",
     "Upgrade actions/cache to v4",
     "sed -i 's|actions/cache@v[23]|actions/cache@v4|g' .github/workflows/*.yml"),
    (r"missing name",
     "Add name: field to workflow YAML",
     "# Add at top of .github/workflows/<file>.yml:\nname: My Workflow"),
    (r"no permissions",
     "Add least-privilege permissions block",
     "permissions:\n  contents: read\n  actions: read"),
    (r"HARDCODED",
     "Move credential to GitHub Actions secret immediately",
     "Replace hardcoded value with ${{ secrets.MY_SECRET }}\nAdd secret at: github.com/<repo>/settings/secrets/actions"),
    (r"unpinned.*requirements|requirements.*unpinned",
     "Pin Python dependency",
     "pip freeze | grep <pkg>  # then add <pkg>==X.Y.Z to requirements.txt"),
    (r"Cannot find module|ModuleNotFoundError",
     "Missing dependency",
     "npm install <module>  # or: pip install <package>"),
]

all_findings = (
    [row[4] for row in AUDIT_TABLE] +
    [row[2] for row in LINT_RESULTS]
)

for finding in all_findings:
    for pattern, label, cmd in FIX_PATTERNS:
        if re.search(pattern, str(finding), re.I):
            entry = (label, cmd)
            if entry not in SUGGESTIONS:
                SUGGESTIONS.append(entry)

if not SUGGESTIONS:
    print("No auto-fix suggestions - everything looks clean.")
else:
    print(f"{len(SUGGESTIONS)} suggestion(s):\n")
    for i, (label, cmd) in enumerate(SUGGESTIONS, 1):
        print(f"{'='*55}")
        print(f"{i}. {label}")
        print("   Fix:")
        for line in cmd.splitlines():
            print(f"     {line}")

In [ ]:
# Cell 9 - Build Health Report - commit to nse-kb/04_reports/
clone_url = f"https://{GITHUB_PAT}@github.com/{PRIMARY_REPO}.git"
if not Path(f"{LOCAL_DIR}/.git").exists():
    if Path(LOCAL_DIR).exists(): shutil.rmtree(LOCAL_DIR)
    gitmod.Repo.clone_from(clone_url, LOCAL_DIR)
    print(f"Cloned {PRIMARY_REPO}")

repo_obj    = gitmod.Repo(LOCAL_DIR)
fname       = f"{DATE_SLUG}__report__build-health-scan__working.md"
report_path = Path(LOCAL_DIR) / "04_reports" / fname
report_path.parent.mkdir(parents=True, exist_ok=True)

def md_table(rows, headers):
    if not rows: return "_No data._\n"
    col_w = [max(len(str(h)), max((len(str(r[i])) for r in rows), default=0))
             for i, h in enumerate(headers)]
    hdr  = "|" + "|".join(str(h).ljust(w) for h, w in zip(headers, col_w)) + "|"
    sep  = "|" + "|".join("-" * w for w in col_w) + "|"
    body = ["|" + "|".join(str(r[i]).ljust(w) for i, w in enumerate(col_w)) + "|"
            for r in rows]
    return "\n".join([hdr, sep] + body) + "\n"

fixes_text = "_No suggestions._"
if SUGGESTIONS:
    lines = []
    for i, (label, cmd) in enumerate(SUGGESTIONS, 1):
        lines.append(f"{i}. **{label}**")
        lines.append("```")
        lines.append(cmd)
        lines.append("```")
    fixes_text = "\n".join(lines)

report = (
    "---\n"
    f"title: Build Health Scan {DATE_SLUG}\n"
    "status: working\n"
    "domain: report\n"
    "freshness: current\n"
    f"last_reviewed: {DATE_SLUG}\n"
    "source_scope: internal-only\n"
    "tags: [report, build-health, ci]\n"
    "confidence: high\n"
    f"owner: {GH_USER}\n"
    "version: 1\n"
    "related: []\n"
    "---\n\n"
    f"# Build Health Scan {DATE_SLUG}\n\n"
    f"Generated: {TIMESTAMP} | Repos: {len(ALL_REPOS)}\n\n"
    "## CI Status\n\n"
    + md_table(ALL_RUNS, ["Repo","Workflow","Result","SHA","Started","URL"]) +
    "\n## Dependency Audit\n\n"
    + (md_table(AUDIT_TABLE, ["Repo","File","Package","Version","Issue"]) if AUDIT_TABLE else "_No issues._") +
    "\n## Workflow Lint\n\n"
    + md_table(LINT_RESULTS, ["Repo","Workflow","Finding","Link"]) +
    "\n## Cross-Repo Health\n\n"
    + md_table(BUILD_HEALTH, ["Repo","Branch","CI","Last Push","Issues","Missing"]) +
    "\n## Auto-Fix Suggestions\n\n"
    + fixes_text + "\n"
)

report_path.write_text(report)
repo_obj.git.add(str(report_path))
if repo_obj.is_dirty(index=True):
    with repo_obj.config_writer() as cw:
        cw.set_value("user", "name", GH_USER)
        cw.set_value("user", "email", f"{GH_USER}@users.noreply.github.com")
    repo_obj.index.commit(f"report(build-health): {DATE_SLUG} [{TIMESTAMP}]")
    repo_obj.remotes.origin.set_url(clone_url)
    repo_obj.remotes.origin.push(BRANCH)
    print(f"Report pushed: 04_reports/{fname}")
    print(f"URL: https://github.com/{PRIMARY_REPO}/blob/{BRANCH}/04_reports/{fname}")
else:
    print("Report unchanged.")

In [ ]:
# Cell 10 - Rerun Failed CI Jobs
if not FAILED_RUNS:
    print("No failed runs to rerun.")
else:
    print(f"Triggering rerun for {len(FAILED_RUNS)} failed run(s)...\n")
    for item in FAILED_RUNS:
        repo, run = item["repo"], item["run"]
        r = requests.post(
            f"{API}/repos/{repo}/actions/runs/{run['id']}/rerun-failed-jobs",
            headers=H
        )
        msgs = {
            201: "Rerun triggered",
            403: "Permission denied - PAT needs Actions: Write",
            409: "Already running"
        }
        print(f"  {msgs.get(r.status_code, f'HTTP {r.status_code}')}: {repo} -> {run['name']}")
        if r.status_code == 201:
            print(f"    Watch: {run['html_url']}")